In [1]:
import sagemaker
import boto3
import os
from sagemaker.workflow.pipeline_context import PipelineSession

# 1. INITIALIZE SESSIONS
# We use PipelineSession specifically for MLOps workflows
sagemaker_session = sagemaker.Session()
pipeline_session = PipelineSession()
region = boto3.Session().region_name
role = sagemaker.get_execution_role()

# 2. CONFIGURE STORAGE LOCATIONS
# Replace 'YOUR-BUCKET-NAME' with the exact name of the S3 bucket you created
data_storage_bucket = "retail-fashion-mlops-data-harivittalmahendrakar" 

# Defining the specific path to your raw fashion data
raw_data_location = f"s3://{data_storage_bucket}/historical_sales.csv"
interaction_data_location = f"s3://{data_storage_bucket}/user_interactions.csv"

# 3. VERIFICATION
print(f"Project Region: {region}")
print(f"Execution Role ARN: {role}")
print(f"Primary Data Source: {raw_data_location}")

# This confirms the folder you are working in
current_directory = os.getcwd()
print(f"Current Working Directory: {current_directory}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Project Region: us-east-1
Execution Role ARN: arn:aws:iam::671835809777:role/service-role/AmazonSageMaker-ExecutionRole-20260413T110336
Primary Data Source: s3://retail-fashion-mlops-data-harivittalmahendrakar/historical_sales.csv
Current Working Directory: /home/sagemaker-user/MLOps_Assignment


In [2]:
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep

# 1. SETUP THE PROCESSOR
# Using a small instance to keep your Free Tier safe
retail_processor = ScriptProcessor(
    command=['python3'],
    image_uri=sagemaker.image_uris.retrieve(framework='sklearn', region=region, version='1.2-1'),
    role=role,
    instance_count=1,
    instance_type='ml.t3.medium',
    sagemaker_session=pipeline_session
)

# 2. DEFINE THE PROCESSING STEP
# This maps your S3 data (raw_data_location) to the preprocess.py script
step_preprocess_retail = ProcessingStep(
    name="FashionDataEngineering",
    processor=retail_processor,
    inputs=[
        ProcessingInput(source=raw_data_location, destination="/opt/ml/processing/input")
    ],
    outputs=[
        ProcessingOutput(output_name="train_data", source="/opt/ml/processing/train")
    ],
    code="preprocess.py" 
)

print("Corrected Cell 2: Data Engineering step is now defined.")

Corrected Cell 2: Data Engineering step is now defined.


In [3]:
from sagemaker.xgboost.estimator import XGBoost
from sagemaker.workflow.steps import TrainingStep
from sagemaker.inputs import TrainingInput

# 1. DEFINE THE ESTIMATOR (The "How-To" for Training)
# We specify 'entry_point' so AWS knows to run your custom train.py script
fashion_model_estimator = XGBoost(
    entry_point="train.py",          # Matches your script filename
    framework_version="1.5-1",       # Stable version for retail forecasting
    instance_type="ml.m5.large",     # Efficient for small retail datasets
    role=role,
    sagemaker_session=pipeline_session,
    instance_count=1,
    output_path=f"s3://{data_storage_bucket}/output"
)

# 2. DEFINE THE TRAINING STEP (The "What" for the Pipeline)
# This links the output of the Preprocessing step to the input of this step
step_train_model = TrainingStep(
    name="FashionDemandTraining",
    estimator=fashion_model_estimator,
    inputs={
        "train": TrainingInput(
            s3_data=step_preprocess_retail.properties.ProcessingOutputConfig.Outputs["train_data"].S3Output.S3Uri,
            content_type="text/csv"
        )
    }
)

print("Training step successfully updated to 'Script Mode'.")

Training step successfully updated to 'Script Mode'.


In [4]:
from sagemaker.workflow.pipeline import Pipeline

# 1. ORCHESTRATE THE STEPS
# This creates the formal pipeline object using the steps we defined
# Note the custom name - this identifies your work in the AWS Console
fashion_retail_pipeline = Pipeline(
    name="FashionDemandMLOpsPipeline",
    steps=[step_preprocess_retail, step_train_model],
    sagemaker_session=pipeline_session
)

# 2. UPLOAD AND REGISTER THE PIPELINE
# This sends the definition to the AWS Cloud
print("Registering the pipeline with AWS SageMaker...")
fashion_retail_pipeline.upsert(role_arn=role)

# 3. VERIFICATION
print("---")
print("SUCCESS: Your Fashion MLOps Pipeline is now live.")
print("Proceed to the 'Pipelines' tab to view and screenshot your DAG.")

Registering the pipeline with AWS SageMaker...


---
SUCCESS: Your Fashion MLOps Pipeline is now live.
Proceed to the 'Pipelines' tab to view and screenshot your DAG.


In [5]:
from sagemaker.workflow.step_collections import RegisterModel

# 1. DEFINE THE REGISTRATION STEP
step_register_model = RegisterModel(
    name="RegisterFashionDemandModel",
    estimator=fashion_model_estimator,
    model_data=step_train_model.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium"],
    transform_instances=["ml.m5.large"],
    model_package_group_name="FashionRetailModelGroup",
    approval_status="PendingManualApproval" # Good for MLOps governance!
)

# 2. UPDATE THE PIPELINE TO INCLUDE ALL 3 STEPS
fashion_retail_pipeline = Pipeline(
    name="FashionDemandMLOpsPipeline",
    steps=[step_preprocess_retail, step_train_model, step_register_model],
    sagemaker_session=pipeline_session
)

# 3. UPSERT AGAIN
fashion_retail_pipeline.upsert(role_arn=role)
print("Pipeline updated with Model Registration. Your DAG will now show 3 boxes.")

Pipeline updated with Model Registration. Your DAG will now show 3 boxes.
